In [12]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

In [2]:
#website to scrap
url="https://quotes.toscrape.com/"

In [4]:
#establish website connection using request then parse the html using beautiful soup
response=requests.get(url)
soup=BeautifulSoup(response.text,"html.parser")

In [5]:
#finding the top 10 tags
tag_section= soup.find("div",class_="col-md-4 tags-box")
tag_links=tag_section.find_all("a",class_="tag")

In [8]:
tags = []

for tag_link in tag_links:
    tag = tag_link.get_text(strip=True)
    tags.append(tag)
print(tags)

['love', 'inspirational', 'life', 'humor', 'books', 'reading', 'friendship', 'friends', 'truth', 'simile']


In [9]:
for tag_link in tag_links:
    print(
        tag_link.get_text(strip=True),
        "->",
        tag_link.get("href")
    )

love -> /tag/love/
inspirational -> /tag/inspirational/
life -> /tag/life/
humor -> /tag/humor/
books -> /tag/books/
reading -> /tag/reading/
friendship -> /tag/friendship/
friends -> /tag/friends/
truth -> /tag/truth/
simile -> /tag/simile/


In [11]:
#list of names and tags
top_tags = []
for tag_link in tag_links:
    tag = tag_link.get_text(strip=True)
    tag_href=tag_link.get("href")
    top_tags.append({
        "Tags":tag,
        "URL":tag_href
         
    })
print(top_tags)    

[{'Tags': 'love', 'URL': '/tag/love/'}, {'Tags': 'inspirational', 'URL': '/tag/inspirational/'}, {'Tags': 'life', 'URL': '/tag/life/'}, {'Tags': 'humor', 'URL': '/tag/humor/'}, {'Tags': 'books', 'URL': '/tag/books/'}, {'Tags': 'reading', 'URL': '/tag/reading/'}, {'Tags': 'friendship', 'URL': '/tag/friendship/'}, {'Tags': 'friends', 'URL': '/tag/friends/'}, {'Tags': 'truth', 'URL': '/tag/truth/'}, {'Tags': 'simile', 'URL': '/tag/simile/'}]


In [13]:
#join href with main URL
for tag in top_tags:
    tag["URL"]=urljoin(url,tag["URL"])

In [16]:
for tag in top_tags:
    print(tag["Tags"],"->",tag["URL"])
    

love -> https://quotes.toscrape.com/tag/love/
inspirational -> https://quotes.toscrape.com/tag/inspirational/
life -> https://quotes.toscrape.com/tag/life/
humor -> https://quotes.toscrape.com/tag/humor/
books -> https://quotes.toscrape.com/tag/books/
reading -> https://quotes.toscrape.com/tag/reading/
friendship -> https://quotes.toscrape.com/tag/friendship/
friends -> https://quotes.toscrape.com/tag/friends/
truth -> https://quotes.toscrape.com/tag/truth/
simile -> https://quotes.toscrape.com/tag/simile/


In [17]:
love_url=top_tags[0]["URL"]
print(love_url)

https://quotes.toscrape.com/tag/love/


In [19]:
response=requests.get(love_url)
love_soup=BeautifulSoup(response.text,"html.parser")

In [20]:
quotes= love_soup.find_all("div",class_="quote")


In [21]:
print(len(quotes))

10


In [22]:
quote = quotes[0]

In [23]:
text=quote.find("span",class_="text").get_text(strip=True)
print(text)

“It is better to be hated for what you are than to be loved for what you are not.”


In [25]:
author = quote.find(
    "small",
    class_="author"
).get_text(strip=True)

print(author)

André Gide


In [26]:
author_link = quote.find(
    "a",
    string="(about)"
)

In [27]:
author_href=author_link.get("href")
print(author_href)

/author/Andre-Gide


In [28]:
author_url=urljoin(url, author_href)
print(author_url)

https://quotes.toscrape.com/author/Andre-Gide


In [30]:
#Extract all quotes on the page(love tag)
quotes_data = []

for quote in quotes:

    text = quote.find(
        "span",
        class_="text"
    ).get_text(strip=True)

    author = quote.find(
        "small",
        class_="author"
    ).get_text(strip=True)

    author_href = quote.find(
        "a",
        string="(about)"
    ).get("href")

    author_url = urljoin(url, author_href)

    quotes_data.append({
        "Quote": text,
        "Author": author,
        "Author URL": author_url
    })

In [31]:
print(quotes_data[0])

{'Quote': '“It is better to be hated for what you are than to be loved for what you are not.”', 'Author': 'André Gide', 'Author URL': 'https://quotes.toscrape.com/author/Andre-Gide'}


In [32]:
print(len(quotes_data))

10


In [33]:
quotes_df = pd.DataFrame(quotes_data)

quotes_df.head()

,Quote,Author,Author URL
0,“It is better to be hated for what you are tha...,André Gide,https://quotes.toscrape.com/author/Andre-Gide
1,“This life is what you make it. No matter what...,Marilyn Monroe,https://quotes.toscrape.com/author/Marilyn-Monroe
2,"“You may not be her first, her last, or her on...",Bob Marley,https://quotes.toscrape.com/author/Bob-Marley
3,"“The opposite of love is not hate, it's indiff...",Elie Wiesel,https://quotes.toscrape.com/author/Elie-Wiesel
4,"“It is not a lack of love, but a lack of frien...",Friedrich Nietzsche,https://quotes.toscrape.com/author/Friedrich-N...


In [35]:
quotes_data = []

current_url = love_url

while current_url:

    print("Scraping:", current_url)

    response = requests.get(current_url)

    love_soup = BeautifulSoup(
        response.text,
        "html.parser"
    )

    quotes = love_soup.find_all(
        "div",
        class_="quote"
    )

    for quote in quotes:

        text = quote.find(
            "span",
            class_="text"
        ).get_text(strip=True)

        author = quote.find(
            "small",
            class_="author"
        ).get_text(strip=True)

        author_href = quote.find(
            "a",
            string="(about)"
        ).get("href")

        author_url = urljoin(
            current_url,
            author_href
        )

        quotes_data.append({
            "Quote": text,
            "Author": author,
            "Author URL": author_url
        })

    next_link = love_soup.find(
        "li",
        class_="next"
    )

    if next_link:

        next_href = next_link.a.get("href")

        current_url = urljoin(
            current_url,
            next_href
        )

    else:

        current_url = None

Scraping: https://quotes.toscrape.com/tag/love/
Scraping: https://quotes.toscrape.com/tag/love/page/2/


In [36]:
print(len(quotes_data))

14


In [37]:
quotes_df = pd.DataFrame(quotes_data)
quotes_df.head()

,Quote,Author,Author URL
0,“It is better to be hated for what you are tha...,André Gide,https://quotes.toscrape.com/author/Andre-Gide
1,“This life is what you make it. No matter what...,Marilyn Monroe,https://quotes.toscrape.com/author/Marilyn-Monroe
2,"“You may not be her first, her last, or her on...",Bob Marley,https://quotes.toscrape.com/author/Bob-Marley
3,"“The opposite of love is not hate, it's indiff...",Elie Wiesel,https://quotes.toscrape.com/author/Elie-Wiesel
4,"“It is not a lack of love, but a lack of frien...",Friedrich Nietzsche,https://quotes.toscrape.com/author/Friedrich-N...


In [40]:
all_quotes = []
for tag_info in top_tags:
    tag=tag = tag_info["Tags"]
    tag_url = tag_info["URL"]
    current_url = tag_url
    while current_url:
        print("Scraping:", current_url)
        response=requests.get(current_url)
        tag_soup=BeautifulSoup(
            response.text,
            "html.parser"
        )
        quotes=tag_soup.find_all(
            "div",
            class_="quote"
        )
        for quote in quotes:
            text=quote.find(
                "span",
                class_="text"
            ).get_text(strip=True)
            author=quote.find(
                "small",
                class_="author"
            ).get_text(strip=True)
            author_href=quote.find(
                "a",
                string="(about)"
            ).get("href")
            author_url=urljoin(
                current_url,
                author_href
            )
            all_quotes.append({
                "Tag":tag,
                "Quote":text,
                "Author":author,
                "Author_url":author_url
                
            })
        next_link=tag_soup.find(
            "li",
            class_="next"
        )
        if next_link:
            next_href=next_link.a.get("href")
            current_url=urljoin(
                current_url,
                next_href
            )
        else:
           current_url=None 

Scraping: https://quotes.toscrape.com/tag/love/
Scraping: https://quotes.toscrape.com/tag/love/page/2/
Scraping: https://quotes.toscrape.com/tag/inspirational/
Scraping: https://quotes.toscrape.com/tag/inspirational/page/2/
Scraping: https://quotes.toscrape.com/tag/life/
Scraping: https://quotes.toscrape.com/tag/life/page/2/
Scraping: https://quotes.toscrape.com/tag/humor/
Scraping: https://quotes.toscrape.com/tag/humor/page/2/
Scraping: https://quotes.toscrape.com/tag/books/
Scraping: https://quotes.toscrape.com/tag/books/page/2/
Scraping: https://quotes.toscrape.com/tag/reading/
Scraping: https://quotes.toscrape.com/tag/friendship/
Scraping: https://quotes.toscrape.com/tag/friends/
Scraping: https://quotes.toscrape.com/tag/truth/
Scraping: https://quotes.toscrape.com/tag/simile/


In [42]:
quotes_df=pd.DataFrame(all_quotes)
quotes_df.tail(10)

,Tag,Quote,Author,Author_url
76,friends,“It takes a great deal of bravery to stand up ...,J.K. Rowling,https://quotes.toscrape.com/author/J-K-Rowling
77,friends,"“Good friends, good books, and a sleepy consci...",Mark Twain,https://quotes.toscrape.com/author/Mark-Twain
78,friends,“There is no friend as loyal as a book.”,Ernest Hemingway,https://quotes.toscrape.com/author/Ernest-Hemi...
79,truth,“The reason I talk to myself is because I’m th...,George Carlin,https://quotes.toscrape.com/author/George-Carlin
80,truth,“A lie can travel half way around the world wh...,Mark Twain,https://quotes.toscrape.com/author/Mark-Twain
81,truth,"“The truth."" Dumbledore sighed. ""It is a beaut...",J.K. Rowling,https://quotes.toscrape.com/author/J-K-Rowling
82,truth,“Never tell the truth to people who are not wo...,Mark Twain,https://quotes.toscrape.com/author/Mark-Twain
83,simile,"“A day without sunshine is like, you know, nig...",Steve Martin,https://quotes.toscrape.com/author/Steve-Martin
84,simile,“Life is like riding a bicycle. To keep your b...,Albert Einstein,https://quotes.toscrape.com/author/Albert-Eins...
85,simile,"“He's like a drug for you, Bella.”",Stephenie Meyer,https://quotes.toscrape.com/author/Stephenie-M...


In [43]:
quotes_df["Tag"].value_counts()

Tag
love             14
inspirational    13
life             13
humor            12
books            11
reading           7
friendship        5
friends           4
truth             4
simile            3
Name: count, dtype: int64

In [46]:
quotes_df["Author"].nunique()

43

In [47]:
#scrape Authors information:
author_url=quotes_df.loc[0,"Author_url"]
print(author_url)

https://quotes.toscrape.com/author/Andre-Gide


In [48]:
response=requests.get(author_url)
author_soup=BeautifulSoup(
    response.text,
    "html.parser"
)

In [50]:
author_details=author_soup.find(
    "div",
    class_="author-details"
)
print(author_details.get_text(" ",strip=True))

André Gide Born: November 22, 1869 in Paris, France Description: André Paul Guillaume Gide was a French author and winner of the Nobel Prize in literature in 1947. Gide's career ranged from its beginnings in the symbolist movement, to the advent of anticolonialism between the two World Wars.Known for his fiction as well as his autobiographical works, Gide exposes to public view the conflict and eventual reconciliation between the two sides of his personality, split apart by a straight-laced education and a narrow social moralism. Gide's work can be seen as an investigation of freedom and empowerment in the face of moralistic and puritan constraints, and gravitates around his continuous effort to achieve intellectual honesty. His self-exploratory texts reflect his search of how to be fully oneself, even to the point of owning one's sexual nature, without at the same time betraying one's values. His political activity is informed by the same ethos, as suggested by his repudiation of comm

In [51]:
born_date=author_details.find(
    "span",
    class_="author-born-date"
).get_text(strip=True)
print(born_date)

November 22, 1869


In [53]:
born_location=author_details.find(
    "span",
    class_="author-born-location"
).get_text(strip=True)
print(born_location)

in Paris, France


In [54]:
author_urls=quotes_df["Author_url"].unique()
print(len(author_urls))

43


In [63]:
authors_data = []
for author_url in author_urls:
    response=requests.get(author_url)
    author_soup=BeautifulSoup(
      response.text,
      "html.parser"
    )
    born_date=author_soup.find(
        "span",
        class_="author-born-date"
    ).get_text(strip=True)
    born_location=author_soup.find(
        "span",
        class_="author-born-location"
    ).get_text(strip=True)
    authors_data.append({
        "Author_url":author_url,
        "Born Date":born_date,
        "Birth Location":born_location
    })
    

In [64]:
authors_df = pd.DataFrame(authors_data)
authors_df.head()

,Author_url,Born Date,Birth Location
0,https://quotes.toscrape.com/author/Andre-Gide,"November 22, 1869","in Paris, France"
1,https://quotes.toscrape.com/author/Marilyn-Monroe,"June 01, 1926",in The United States
2,https://quotes.toscrape.com/author/Bob-Marley,"February 06, 1945","in Nine Mile, Saint Ann, Jamaica"
3,https://quotes.toscrape.com/author/Elie-Wiesel,"September 30, 1928","in Sighet, Romania"
4,https://quotes.toscrape.com/author/Friedrich-N...,"October 15, 1844","in Röcken bei Lützen, Prussian Province of Sax..."


In [69]:
quotes_df = quotes_df.merge(
    authors_df,
    on="Author_url",
    how="left"
)

In [66]:
print(quotes_df.columns.tolist())

['Tag', 'Quote', 'Author', 'Author_url']


In [68]:
print(authors_df.columns.tolist())

['Author_url', 'Born Date', 'Birth Location']


In [70]:
quotes_df.head()

,Tag,Quote,Author,Author_url,Born Date,Birth Location
0,love,“It is better to be hated for what you are tha...,André Gide,https://quotes.toscrape.com/author/Andre-Gide,"November 22, 1869","in Paris, France"
1,love,“This life is what you make it. No matter what...,Marilyn Monroe,https://quotes.toscrape.com/author/Marilyn-Monroe,"June 01, 1926",in The United States
2,love,"“You may not be her first, her last, or her on...",Bob Marley,https://quotes.toscrape.com/author/Bob-Marley,"February 06, 1945","in Nine Mile, Saint Ann, Jamaica"
3,love,"“The opposite of love is not hate, it's indiff...",Elie Wiesel,https://quotes.toscrape.com/author/Elie-Wiesel,"September 30, 1928","in Sighet, Romania"
4,love,"“It is not a lack of love, but a lack of frien...",Friedrich Nietzsche,https://quotes.toscrape.com/author/Friedrich-N...,"October 15, 1844","in Röcken bei Lützen, Prussian Province of Sax..."


In [71]:
print(quotes_df.shape)
print(authors_df.shape)

(86, 6)
(43, 3)


In [73]:
quotes_df[[
    "Author",
    "Born Date",
    "Birth Location"
]].head(10)

,Author,Born Date,Birth Location
0,André Gide,"November 22, 1869","in Paris, France"
1,Marilyn Monroe,"June 01, 1926",in The United States
2,Bob Marley,"February 06, 1945","in Nine Mile, Saint Ann, Jamaica"
3,Elie Wiesel,"September 30, 1928","in Sighet, Romania"
4,Friedrich Nietzsche,"October 15, 1844","in Röcken bei Lützen, Prussian Province of Sax..."
5,Pablo Neruda,"July 12, 1904","in Parral, Chile"
6,Marilyn Monroe,"June 01, 1926",in The United States
7,Marilyn Monroe,"June 01, 1926",in The United States
8,James Baldwin,"August 02, 1924","in Harlem, New York, The United States"
9,Jane Austen,"December 16, 1775","in Steventon Rectory, Hampshire, The United Ki..."
